<a href="https://colab.research.google.com/github/AmitavaDutta/parallel_data_processing_engine/blob/main/Bandwidth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import time

def measure_pcie_bandwidth(size_gb=1):
    # Calculate number of float32 elements for 1GB
    # 1GB = 1024^3 bytes. float32 = 4 bytes.
    n_elements = (size_gb * 1024**3) // 4

    # 1. Create Pinned Memory on CPU
    data_cpu = torch.randn(n_elements, dtype=torch.float32).pin_memory()

    # 2. Measure Host-to-Device (H2D)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    data_gpu = data_cpu.to('cuda', non_blocking=True)
    torch.cuda.synchronize()
    t1 = time.perf_counter()

    h2d_time = t1 - t0
    h2d_bw = size_gb / h2d_time

    # 3. Measure Device-to-Host (D2H)
    torch.cuda.synchronize()
    t2 = time.perf_counter()
    data_back = data_gpu.to('cpu', non_blocking=True)
    torch.cuda.synchronize()
    t3 = time.perf_counter()

    d2h_time = t3 - t2
    d2h_bw = size_gb / d2h_time

    print(f"H2D Bandwidth: {h2d_bw:.2f} GB/s")
    print(f"D2H Bandwidth: {d2h_bw:.2f} GB/s")

measure_pcie_bandwidth()

H2D Bandwidth: 10.58 GB/s
D2H Bandwidth: 1.48 GB/s
